# Customer Churn Analytics Platform

## 02 — Data Cleaning & Data Quality

**Objective:**
Clean, standardize, and validate the customer churn dataset while preserving valid customer records and business meaning.

### Cleaning Goals

* Load the raw dataset
* Create a clean working dataset
* Standardize column names
* Remove unnecessary text spaces
* Identify blank and missing values
* Convert numerical fields to appropriate data types
* Handle missing `TotalCharges` values
* Validate categorical values
* Validate numerical fields
* Check duplicate records
* Validate unique customer IDs
* Perform final data quality checks
* Save the cleaned dataset for further analysis


In [1]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# ============================================================
# LOAD RAW DATASET
# ============================================================

DATA_PATH = Path("../data/raw/Telco-Customer-Churn.csv")

df = pd.read_csv(DATA_PATH)

print("Raw dataset loaded successfully.")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]:,}")

Raw dataset loaded successfully.
Rows    : 7,043
Columns : 21


In [3]:
# ============================================================
# CREATE WORKING COPY
# ============================================================

df_clean = df.copy()

print("Working copy created successfully.")

Working copy created successfully.


In [4]:
# ============================================================
# STANDARDIZE COLUMN NAMES
# ============================================================

df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("Column names standardized.")

print("\nColumns:")
print(df_clean.columns.tolist())

Column names standardized.

Columns:
['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents', 'tenure', 'phoneservice', 'multiplelines', 'internetservice', 'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod', 'monthlycharges', 'totalcharges', 'churn']


In [5]:
# ============================================================
# CLEAN TEXT SPACES
# ============================================================

text_columns = df_clean.select_dtypes(
    include="object"
).columns

for column in text_columns:
    df_clean[column] = (
        df_clean[column]
        .str.strip()
    )

print("Text fields cleaned successfully.")

Text fields cleaned successfully.


C:\Users\Kunde\AppData\Local\Temp\ipykernel_9308\3186729072.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df_clean.select_dtypes(


In [6]:
# ============================================================
# CHECK BLANK VALUES
# ============================================================

blank_summary = pd.DataFrame({
    "Blank_Count": [
        (df_clean[column].astype(str).str.strip() == "").sum()
        for column in df_clean.columns
    ]
}, index=df_clean.columns)

blank_summary = blank_summary[
    blank_summary["Blank_Count"] > 0
]

blank_summary

,Blank_Count
totalcharges,11


In [7]:
# ============================================================
# CONVERT TOTALCHARGES TO NUMERIC
# ============================================================

df_clean["totalcharges"] = pd.to_numeric(
    df_clean["totalcharges"],
    errors="coerce"
)

print("TotalCharges converted to numeric.")

print("\nData Type:")
print(df_clean["totalcharges"].dtype)

print("\nMissing TotalCharges after conversion:")
print(df_clean["totalcharges"].isna().sum())

TotalCharges converted to numeric.

Data Type:
float64

Missing TotalCharges after conversion:
11


In [8]:
# ============================================================
# INSPECT MISSING TOTALCHARGES
# ============================================================

missing_totalcharges = df_clean[
    df_clean["totalcharges"].isna()
][
    [
        "customerid",
        "tenure",
        "monthlycharges",
        "totalcharges",
        "churn"
    ]
]

print(f"Records with missing TotalCharges: {len(missing_totalcharges)}")

missing_totalcharges

Records with missing TotalCharges: 11


,customerid,tenure,monthlycharges,totalcharges,churn
488,4472-LVYGI,0,52.55,NaN,No
753,3115-CZMZD,0,20.25,NaN,No
936,5709-LVOEQ,0,80.85,NaN,No
1082,4367-NUYAO,0,25.75,NaN,No
1340,1371-DWPAZ,0,56.05,NaN,No
3331,7644-OMVMY,0,19.85,NaN,No
3826,3213-VVOLG,0,25.35,NaN,No
4380,2520-SGTTA,0,20.00,NaN,No
5218,2923-ARZLG,0,19.70,NaN,No
6670,4075-WKNIU,0,73.35,NaN,No


In [9]:
# ============================================================
# VALIDATE MISSING TOTALCHARGES RECORDS
# ============================================================

missing_check = df_clean[
    df_clean["totalcharges"].isna()
]

print("Missing TotalCharges:", len(missing_check))
print("Tenure = 0:", (missing_check["tenure"] == 0).sum())
print(
    "MonthlyCharges > 0:",
    (missing_check["monthlycharges"] > 0).sum()
)

print("\nTenure values:")
print(missing_check["tenure"].value_counts())

Missing TotalCharges: 11
Tenure = 0: 11
MonthlyCharges > 0: 11

Tenure values:
tenure
0    11
Name: count, dtype: int64


In [10]:
# ============================================================
# HANDLE MISSING TOTALCHARGES
# ============================================================

missing_before = df_clean["totalcharges"].isna().sum()

df_clean.loc[
    (df_clean["totalcharges"].isna()) &
    (df_clean["tenure"] == 0),
    "totalcharges"
] = 0.0

missing_after = df_clean["totalcharges"].isna().sum()

print("Missing TotalCharges handled.")

print(f"Missing before : {missing_before}")
print(f"Missing after  : {missing_after}")
print(
    "Records filled :",
    missing_before - missing_after
)

Missing TotalCharges handled.
Missing before : 11
Missing after  : 0
Records filled : 11


In [11]:
# ============================================================
# VALIDATE TOTALCHARGES
# ============================================================

print("TotalCharges validation")
print("=" * 50)

print(f"Data Type       : {df_clean['totalcharges'].dtype}")
print(f"Missing Values  : {df_clean['totalcharges'].isna().sum()}")
print(f"Minimum         : {df_clean['totalcharges'].min():.2f}")
print(f"Maximum         : {df_clean['totalcharges'].max():.2f}")
print(f"Negative Values : {(df_clean['totalcharges'] < 0).sum()}")

TotalCharges validation
Data Type       : float64
Missing Values  : 0
Minimum         : 0.00
Maximum         : 8684.80
Negative Values : 0


In [12]:
# ============================================================
# DATA TYPE VALIDATION
# ============================================================

dtype_summary = pd.DataFrame({
    "Data_Type": df_clean.dtypes.astype(str),
    "Missing": df_clean.isna().sum(),
    "Unique": df_clean.nunique()
})

dtype_summary

,Data_Type,Missing,Unique
customerid,str,0,7043
gender,str,0,2
seniorcitizen,int64,0,2
partner,str,0,2
dependents,str,0,2
tenure,int64,0,73
phoneservice,str,0,2
multiplelines,str,0,3
internetservice,str,0,3
onlinesecurity,str,0,3


In [13]:
# ============================================================
# DUPLICATE VALIDATION
# ============================================================

duplicate_rows = df_clean.duplicated().sum()

print("Duplicate row validation")
print("=" * 50)
print(f"Duplicate Rows: {duplicate_rows}")

Duplicate row validation
Duplicate Rows: 0


In [14]:
# ============================================================
# CUSTOMER ID VALIDATION
# ============================================================

total_customers = len(df_clean)
unique_customer_ids = df_clean["customerid"].nunique()
missing_customer_ids = df_clean["customerid"].isna().sum()

print("Customer ID validation")
print("=" * 50)

print(f"Total Records       : {total_customers:,}")
print(f"Unique Customer IDs : {unique_customer_ids:,}")
print(f"Missing Customer IDs: {missing_customer_ids}")
print(
    f"IDs Unique          : "
    f"{total_customers == unique_customer_ids}"
)

Customer ID validation
Total Records       : 7,043
Unique Customer IDs : 7,043
Missing Customer IDs: 0
IDs Unique          : True


In [15]:
# ============================================================
# STANDARDIZE CHURN
# ============================================================

df_clean["churn"] = (
    df_clean["churn"]
    .astype(str)
    .str.strip()
    .str.lower()
)

print("Churn values standardized.")

print("\nUnique Churn Values:")
print(df_clean["churn"].unique())

Churn values standardized.

Unique Churn Values:
<ArrowStringArray>
['no', 'yes']
Length: 2, dtype: str


In [16]:
# ============================================================
# CATEGORICAL VALIDATION
# ============================================================

categorical_columns = [
    "gender",
    "partner",
    "dependents",
    "phoneservice",
    "multiplelines",
    "internetservice",
    "onlinesecurity",
    "onlinebackup",
    "deviceprotection",
    "techsupport",
    "streamingtv",
    "streamingmovies",
    "contract",
    "paperlessbilling",
    "paymentmethod",
    "churn"
]

for column in categorical_columns:
    print("\n" + "=" * 60)
    print(f"{column}")
    print("=" * 60)
    print(df_clean[column].value_counts(dropna=False))


gender
gender
Male      3555
Female    3488
Name: count, dtype: int64

partner
partner
No     3641
Yes    3402
Name: count, dtype: int64

dependents
dependents
No     4933
Yes    2110
Name: count, dtype: int64

phoneservice
phoneservice
Yes    6361
No      682
Name: count, dtype: int64

multiplelines
multiplelines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64

internetservice
internetservice
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

onlinesecurity
onlinesecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64

onlinebackup
onlinebackup
No                     3088
Yes                    2429
No internet service    1526
Name: count, dtype: int64

deviceprotection
deviceprotection
No                     3095
Yes                    2422
No internet service    1526
Name: count, dtype: int64

techsupport
techsupport
No             

In [17]:
# ============================================================
# NUMERICAL VALIDATION
# ============================================================

numerical_columns = [
    "seniorcitizen",
    "tenure",
    "monthlycharges",
    "totalcharges"
]

numerical_validation = pd.DataFrame({
    "Min": df_clean[numerical_columns].min(),
    "Max": df_clean[numerical_columns].max(),
    "Mean": df_clean[numerical_columns].mean(),
    "Missing": df_clean[numerical_columns].isna().sum(),
    "Negative": [
        (df_clean[column] < 0).sum()
        for column in numerical_columns
    ]
})

numerical_validation

,Min,Max,Mean,Missing,Negative
seniorcitizen,0.00,1.00,0.162147,0,0
tenure,0.00,72.00,32.371149,0,0
monthlycharges,18.25,118.75,64.761692,0,0
totalcharges,0.00,8684.80,2279.734304,0,0


In [18]:
# ============================================================
# BUSINESS LOGIC VALIDATION
# ============================================================

print("BUSINESS LOGIC VALIDATION")
print("=" * 60)

print(
    "Negative Tenure:",
    (df_clean["tenure"] < 0).sum()
)

print(
    "Tenure > 72:",
    (df_clean["tenure"] > 72).sum()
)

print(
    "Negative MonthlyCharges:",
    (df_clean["monthlycharges"] < 0).sum()
)

print(
    "Negative TotalCharges:",
    (df_clean["totalcharges"] < 0).sum()
)

print(
    "Invalid SeniorCitizen:",
    (~df_clean["seniorcitizen"].isin([0, 1])).sum()
)

print(
    "Invalid Churn:",
    (~df_clean["churn"].isin(["yes", "no"])).sum()
)

BUSINESS LOGIC VALIDATION
Negative Tenure: 0
Tenure > 72: 0
Negative MonthlyCharges: 0
Negative TotalCharges: 0
Invalid SeniorCitizen: 0
Invalid Churn: 0


In [19]:
# ============================================================
# FINAL MISSING VALUE CHECK
# ============================================================

final_missing = df_clean.isna().sum()

final_missing = final_missing[
    final_missing > 0
]

print("FINAL MISSING VALUE CHECK")
print("=" * 60)

if final_missing.empty:
    print("No missing values found.")
else:
    print(final_missing)

FINAL MISSING VALUE CHECK
No missing values found.


In [20]:
# ============================================================
# FINAL DUPLICATE CHECK
# ============================================================

final_duplicates = df_clean.duplicated().sum()

print("FINAL DUPLICATE CHECK")
print("=" * 60)
print(f"Duplicate Rows: {final_duplicates}")

FINAL DUPLICATE CHECK
Duplicate Rows: 0


In [ ]:
# ============================================================
# FINAL DATASET SHAPE
# ============================================================

print("FINAL CLEAN DATASET")
print("=" * 60)

print(f"Rows    : {df_clean.shape[0]:,}")
print(f"Columns : {df_clean.shape[1]:,}")

FINAL CLEAN DATASET
Rows    : 7,043
Columns : 21


In [22]:
# ============================================================
# FINAL QUALITY SUMMARY
# ============================================================

print("=" * 70)
print("CUSTOMER CHURN ANALYTICS — DATA CLEANING SUMMARY")
print("=" * 70)

print(f"Rows                    : {len(df_clean):,}")
print(f"Columns                 : {len(df_clean.columns):,}")
print(f"Unique Customer IDs     : {df_clean['customerid'].nunique():,}")
print(f"Duplicate Rows          : {df_clean.duplicated().sum():,}")
print(f"Missing Values          : {df_clean.isna().sum().sum():,}")
print(f"Negative Tenure         : {(df_clean['tenure'] < 0).sum():,}")
print(f"Negative MonthlyCharges : {(df_clean['monthlycharges'] < 0).sum():,}")
print(f"Negative TotalCharges   : {(df_clean['totalcharges'] < 0).sum():,}")
print(f"Invalid Churn Values    : {(~df_clean['churn'].isin(['yes', 'no'])).sum():,}")

print("=" * 70)

CUSTOMER CHURN ANALYTICS — DATA CLEANING SUMMARY
Rows                    : 7,043
Columns                 : 21
Unique Customer IDs     : 7,043
Duplicate Rows          : 0
Missing Values          : 0
Negative Tenure         : 0
Negative MonthlyCharges : 0
Negative TotalCharges   : 0
Invalid Churn Values    : 0


In [23]:
# ============================================================
# SAVE CLEAN DATASET
# ============================================================

OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "customer_churn_clean.csv"

df_clean.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Clean dataset saved successfully.")
print(f"Path: {OUTPUT_PATH}")
print(f"Rows: {df_clean.shape[0]:,}")
print(f"Columns: {df_clean.shape[1]:,}")

Clean dataset saved successfully.
Path: ..\data\processed\customer_churn_clean.csv
Rows: 7,043
Columns: 21


In [24]:
# ============================================================
# RELOAD VALIDATION
# ============================================================

df_check = pd.read_csv(OUTPUT_PATH)

print("Reload validation completed.")
print("=" * 60)

print(f"Rows             : {df_check.shape[0]:,}")
print(f"Columns          : {df_check.shape[1]:,}")
print(f"Missing Values   : {df_check.isna().sum().sum():,}")
print(f"Duplicate Rows   : {df_check.duplicated().sum():,}")
print(
    f"Unique Customers : "
    f"{df_check['customerid'].nunique():,}"
)

print("=" * 60)

Reload validation completed.
Rows             : 7,043
Columns          : 21
Missing Values   : 0
Duplicate Rows   : 0
Unique Customers : 7,043
